# TLM-S RL Fine-Tuning (Self-Critical Sequence Training) -- Colab


## Cell 1 -- Imports & Device

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys, os, re, json, math, random
from pathlib import Path
import copy as _copy

DRIVE_ROOT = Path('/content/drive/MyDrive/TLM_V03')
assert DRIVE_ROOT.exists(), f"{DRIVE_ROOT} not found -- upload the project to Drive first (see markdown above)"
sys.path.insert(0, str(DRIVE_ROOT))

from src.model_config import ModelConfig, count_parameters
from src.model_variants import TLM_S
from src.tokenizer import TLM_Tokenizer
from src.data_set import make_sft_sr, apply_sr, split_lines_by_newline, join_lines
from src.data_loaders import SFTDataset, collate_sft
from src.training import validate, sample_and_score, self_critical_step, ParallelRolloutCollector
from src.eval import TASK_SCORERS
from src.inference import greedy_decode

import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")


## Cell 2 -- Paths & Run Config


In [ ]:
FINETUNE_DIR = DRIVE_ROOT / "data" / "finetune_data"
SFT_TRAIN_PATH = FINETUNE_DIR / "sft_train_domain.jsonl"   # built by the SFT notebook's Cell 2b -- reuse, don't regenerate
SFT_VAL_PATH   = FINETUNE_DIR / "sft_val_domain.jsonl"
assert SFT_TRAIN_PATH.exists() and SFT_VAL_PATH.exists(), (
    "sft_train_domain.jsonl / sft_val_domain.jsonl not found -- run the SFT notebook's "
    "Cell 2b (same-domain split) at least once, or point these paths elsewhere.")

TOKENIZER_PATH = str(DRIVE_ROOT / "src" / "tokenizer" / "tokenizer_tlm_small.json")

# -- MODEL VARIANT -- set every run -- must match the SFT checkpoint you're loading
MODEL_VARIANT = "S"   # "S" or "M" -- see the project spec's model ladder table
assert MODEL_VARIANT in ("S", "M"), f"MODEL_VARIANT must be 'S' or 'M', got {MODEL_VARIANT!r}"
_variant_prefix = f"tlm_{MODEL_VARIANT.lower()}"

_POLICY_SUBDIR = {"S": "tlm_s_sft_coverage", "M": "tlm_m_sft"}[MODEL_VARIANT]
POLICY_CKPT_DIR = DRIVE_ROOT / "data" / "checkpoints" / _POLICY_SUBDIR

# -- RUN CONFIG -- set every run --
_RUN_NAME_BY_VARIANT = {
    "S": "tlm_s_rl_final",
    "M": "tlm_m_rl_final",
}
RUN_NAME = _RUN_NAME_BY_VARIANT[MODEL_VARIANT]
RUN_MODE = "fresh"    # "fresh" (from POLICY_CKPT_DIR) or "resume" (continue RUN_NAME's own last.pt)
assert RUN_MODE in ("fresh", "resume"), f"RUN_MODE must be 'fresh' or 'resume', got {RUN_MODE!r}"

RL_CKPT_DIR = DRIVE_ROOT / "data" / "checkpoints" / RUN_NAME
RL_CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_VARIANT={MODEL_VARIANT!r}  RUN_NAME={RUN_NAME!r}  RUN_MODE={RUN_MODE!r}")
print(f"  policy from : {POLICY_CKPT_DIR}")
print(f"  checkpoints to: {RL_CKPT_DIR}")


## Cell 3 -- Model, Tokenizer, Policy + Frozen Reference


In [ ]:
if MODEL_VARIANT == "XXS":
    config = ModelConfig(d_model=96, d_ff=384, n_encoder_layers=3, n_decoder_layers=3,
                             n_heads=3, tier_name="TLM_XXS", target_param_count=1_740_000)
elif MODEL_VARIANT == "XS":
    config = ModelConfig(d_model=128, d_ff=512, tier_name="TLM_XS", target_param_count=2_420_000)
elif MODEL_VARIANT == "S":
    config = ModelConfig()
else:  # "M" -- 256 d_model / 1024 d_ff, same 4+4 layers / 4 heads / 6 task types as S
    config = ModelConfig(d_model=256, d_ff=1024, tier_name="TLM_M", target_param_count=9_500_000)
print(config)

PAD_ID = config.special_token_ids["pad"]

tokenizer = TLM_Tokenizer(vocab_size=config.vocab_size)
tokenizer.load(TOKENIZER_PATH)
print(f"Tokenizer loaded  |  vocab_size = {tokenizer.tokenizer.get_vocab_size()}")

model = TLM_S(config)   # class is architecture-agnostic -- entirely parameterized by config


def _val_loss_from_name(p):
    m = re.search(r"val([0-9.]+)\.pt$", p.name)
    return float(m.group(1)) if m else float("inf")


def _reward_from_name(p):
    m = re.search(r"reward([0-9.]+)\.pt$", p.name)
    return float(m.group(1)) if m else float("-inf")


_val_ckpts = sorted(POLICY_CKPT_DIR.glob(f"{POLICY_CKPT_DIR.name}_epoch_*_val*.pt"))
_reward_ckpts = sorted(POLICY_CKPT_DIR.glob(f"{POLICY_CKPT_DIR.name}_epoch_*_reward*.pt"))
assert _val_ckpts or _reward_ckpts, f"No checkpoints found in {POLICY_CKPT_DIR}"
if _reward_ckpts:
    POLICY_CKPT_PATH = max(_reward_ckpts, key=_reward_from_name)
else:
    POLICY_CKPT_PATH = min(_val_ckpts, key=_val_loss_from_name)
_state = torch.load(POLICY_CKPT_PATH, map_location=device)
model.load_state_dict(_state["model"])
model.to(device)
if "val_reward" in _state:
    print(f"Loaded policy start: {POLICY_CKPT_PATH.name}  "
          f"(val_reward={_state['val_reward']:.4f} under the OLD scorers -- "
          f"see Cell 6 below for this run's real, current-scorer baseline)")
else:
    print(f"Loaded policy start: {POLICY_CKPT_PATH.name}  "
          f"(val_nll={_state.get('val_nll'):.4f}, val_pgen={_state.get('val_pgen'):.3f})")

ref_model = _copy.deepcopy(model)
ref_model.to(device)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)
print("Frozen KL-reference model ready (copy of the starting policy).")


## Cell 4 -- RL Data (small pilot subset)


In [4]:
from collections import defaultdict

N_PER_TASK_TRAIN = 300   # data-scale sanity check (Phase 3 #8) -- intermediate value
                         # before committing to the full 300/task run; 5 tasks x 150 = 750
                         # rollout records, ~187 opt steps/epoch at ROLLOUTS_PER_STEP=4
N_PER_TASK_VAL   = 150   # for the per-epoch validation-reward check

def _load_by_type(path):
    by_type = defaultdict(list)
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            by_type[d["type"]].append(d)
    return by_type

_train_by_type = _load_by_type(SFT_TRAIN_PATH)
_val_by_type   = _load_by_type(SFT_VAL_PATH)

_rng = random.Random(SEED)
train_records, val_records = [], []
for ttype in sorted(_train_by_type):
    pool = _train_by_type[ttype][:]
    _rng.shuffle(pool)
    train_records.extend(pool[:N_PER_TASK_TRAIN])
for ttype in sorted(_val_by_type):
    pool = _val_by_type[ttype][:]
    _rng.shuffle(pool)
    val_records.extend(pool[:N_PER_TASK_VAL])

print(f"train_records: {len(train_records)}  val_records: {len(val_records)}")


## Cell 5 -- Training Setup


In [14]:
EPOCHS            = 7
LR                = 3e-6   
ROLLOUTS_PER_STEP = 4      
TEMPERATURE       = 0.5    
MAX_NEW_TOKENS    = 512    
KL_COEF           = 0   
PGEN_LAMBDA_RL    = 0.1    
                           
GAMMA             = 0.3    
                           
                           
                           
CONF_LAMBDA       = 0.1    
USE_PARALLEL_CPU = False
N_PARALLEL_WORKERS = ROLLOUTS_PER_STEP  

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(f"epochs={EPOCHS}  lr={LR}  rollouts/step={ROLLOUTS_PER_STEP}  "
      f"temperature={TEMPERATURE}  kl_coef={KL_COEF}  pgen_lambda_rl={PGEN_LAMBDA_RL}  "
      f"gamma={GAMMA}  conf_lambda={CONF_LAMBDA}")
print(f"use_parallel_cpu={USE_PARALLEL_CPU and device.type == 'cpu'}  "
      f"n_parallel_workers={N_PARALLEL_WORKERS}")


## Cell 6 -- Validation Reward (greedy, real scorer)


In [6]:
@torch.no_grad()
def validate_reward(model, records, tokenizer, config, device, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    scores_by_task = defaultdict(list)
    for d in records:
        res = make_sft_sr(d, tokenizer, config)
        if res is None:
            continue
        enc_ids, _, task_id = res
        before_ids = tokenizer.encode(d["before"])
        after_ids  = tokenizer.encode(d["after"])
        nl_id = config.special_token_ids["newline"]
        gold_after = join_lines(split_lines_by_newline(after_ids, nl_id), nl_id)

        out = greedy_decode(model, enc_ids, task_id, device,
                             max_new_tokens=max_new_tokens, collect_conf=False)
        pred = apply_sr(before_ids, out["tokens"], config)
        before_text = tokenizer.decode(before_ids)
        gold_text = tokenizer.decode(gold_after)
        pred_text = tokenizer.decode(pred) if pred is not None else None
        score = TASK_SCORERS[d["type"]](before_text, pred_text, gold_text)["score"]
        scores_by_task[d["type"]].append(score)

    per_task = {t: sum(s) / len(s) for t, s in scores_by_task.items() if s}
    overall = sum(per_task.values()) / max(len(per_task), 1)
    return overall, per_task


start_val_reward, start_val_per_task = validate_reward(model, val_records, tokenizer, config, device)
print(f"Starting policy val reward: {start_val_reward:.4f}")
for t, s in sorted(start_val_per_task.items()):
    print(f"  {t:20s} {s:.4f}")


## Cell 7 -- Train Loop (resumable)


In [16]:
RESUME_PATH = RL_CKPT_DIR / "last.pt"
COLLAPSE_DROP = 0.10   

history = []
best_val_reward = start_val_reward
best_ckpt_path = None
start_epoch = 1

if RUN_MODE == "resume":
    assert RESUME_PATH.exists(), f"RUN_MODE='resume' but no last.pt at {RESUME_PATH}"
    _ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(_ckpt["model"])
    optimizer.load_state_dict(_ckpt["optimizer"])
    start_epoch = _ckpt["epoch"] + 1
    best_val_reward = _ckpt["best_val_reward"]
    best_ckpt_path = Path(_ckpt["best_ckpt_path"]) if _ckpt.get("best_ckpt_path") else None
    history = _ckpt.get("history", [])
    for _g in optimizer.param_groups:
        _g["lr"] = LR
    print(f"Resuming {RUN_NAME!r} from epoch {start_epoch}  (lr overridden to {LR})")


def _opt_step(model, pending, optimizer, device, pad_id, ref_model, kl_coef, pgen_lambda, gamma, conf_lambda):
    optimizer.zero_grad()
    stats = self_critical_step(model, pending, device, pad_id,
                                ref_model=(ref_model if kl_coef > 0 else None),
                                kl_coef=kl_coef, pgen_lambda=pgen_lambda, gamma=gamma,
                                conf_lambda=conf_lambda)
    stats["loss"].backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return stats


_use_parallel = USE_PARALLEL_CPU and device.type == "cpu"
_collector = None
if _use_parallel:
    _collector = ParallelRolloutCollector(
        model, TOKENIZER_PATH, config, TASK_SCORERS,
        make_sft_sr, apply_sr, split_lines_by_newline, join_lines,
        temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS,
        n_workers=N_PARALLEL_WORKERS,
    )
    print(f"ParallelRolloutCollector active: {N_PARALLEL_WORKERS} workers")


def _print_step_log(epoch, n_opt_steps, stats, reward_s_window, reward_g_window, adv_window, tail_n=None):
    
    tail_tag = f" (tail, {tail_n} rollouts)" if tail_n is not None else ""
    print(f"  epoch {epoch}  step {n_opt_steps:4d}{tail_tag}  "
          f"reward_sample {sum(reward_s_window) / len(reward_s_window):.3f}  "
          f"reward_greedy {sum(reward_g_window) / len(reward_g_window):.3f}  "
          f"advantage {sum(adv_window) / len(adv_window):+.3f}  "
          f"conf_sample {stats['mean_confidence_sample']:.3f}  "
          f"kl {stats['mean_kl']:.4f}  "
          f"pgen_pen {stats['mean_pgen_penalty']:.4f}  "
          f"conf_pen {stats['mean_conf_penalty']:.4f}  "
          f"reinforce {stats['mean_reinforce_loss']:+.4f}  "
          f"pgen_term {stats['mean_pgen_term_weighted']:.4f}  "
          f"conf_term {stats['mean_conf_term_weighted']:.4f}  "
          f"mle_nll {stats['mean_mle_nll']:.4f}  "
          f"mle_term {stats['mean_mle_term_weighted']:.4f}", flush=True)


collapsed = False
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    order = train_records[:]
    random.Random(SEED + epoch).shuffle(order)

    step_rewards_s, step_rewards_g, step_adv = [], [], []
    n_opt_steps = 0

    if _use_parallel:
        
        chunks = [order[i:i + ROLLOUTS_PER_STEP] for i in range(0, len(order), ROLLOUTS_PER_STEP)]
        for pending in map(_collector.collect, chunks):
            if not pending:
                continue
            stats = _opt_step(model, pending, optimizer, device, PAD_ID, ref_model, KL_COEF, PGEN_LAMBDA_RL, GAMMA, CONF_LAMBDA)
            n_opt_steps += 1
            step_rewards_s.append(stats["mean_reward_sample"])
            step_rewards_g.append(stats["mean_reward_greedy"])
            step_adv.append(stats["mean_advantage"])
            if n_opt_steps % 10 == 0:
                _print_step_log(epoch, n_opt_steps, stats,
                                 step_rewards_s[-10:], step_rewards_g[-10:], step_adv[-10:])
    else:
        pending = []
        for i, d in enumerate(order):
            r = sample_and_score(model, d, tokenizer, config, device, TASK_SCORERS,
                                  make_sft_sr, apply_sr, split_lines_by_newline, join_lines,
                                  temperature=TEMPERATURE, max_new_tokens=MAX_NEW_TOKENS)
            if r is None:
                continue
            pending.append(r)
            if len(pending) >= ROLLOUTS_PER_STEP:
                stats = _opt_step(model, pending, optimizer, device, PAD_ID, ref_model, KL_COEF, PGEN_LAMBDA_RL, GAMMA, CONF_LAMBDA)
                n_opt_steps += 1
                step_rewards_s.append(stats["mean_reward_sample"])
                step_rewards_g.append(stats["mean_reward_greedy"])
                step_adv.append(stats["mean_advantage"])
                if n_opt_steps % 10 == 0:
                    _print_step_log(epoch, n_opt_steps, stats,
                                     step_rewards_s[-10:], step_rewards_g[-10:], step_adv[-10:])
                pending = []


        if pending:
            stats = _opt_step(model, pending, optimizer, device, PAD_ID, ref_model, KL_COEF, PGEN_LAMBDA_RL, GAMMA, CONF_LAMBDA)
            n_opt_steps += 1
            step_rewards_s.append(stats["mean_reward_sample"])
            step_rewards_g.append(stats["mean_reward_greedy"])
            step_adv.append(stats["mean_advantage"])
            _print_step_log(epoch, n_opt_steps, stats,
                             [stats["mean_reward_sample"]], [stats["mean_reward_greedy"]],
                             [stats["mean_advantage"]], tail_n=len(pending))

    val_reward, val_per_task = validate_reward(model, val_records, tokenizer, config, device)
    mean_adv = sum(step_adv) / max(len(step_adv), 1)
    print(f"epoch {epoch}: mean_advantage {mean_adv:+.4f} | val_reward {val_reward:.4f} "
          f"(start {start_val_reward:.4f})")
    history.append({"epoch": epoch, "mean_advantage": mean_adv, "val_reward": val_reward,
                     "val_per_task": val_per_task})

    if val_reward < start_val_reward - COLLAPSE_DROP:
        print(f"COLLAPSE GUARD TRIPPED: val_reward {val_reward:.4f} is more than "
              f"{COLLAPSE_DROP} below the starting policy's {start_val_reward:.4f}. "
              f"Stopping -- same signature as Try #3's regression. Do not keep training "
              f"past this point; inspect qualitative samples (Cell 9) before deciding next steps. "
              f"NOT updating last.pt/resume state, so a later RUN_MODE='resume' can't "
              f"silently continue from the collapsed state (the collapsed epoch's weights are "
              f"still saved below under their own per-epoch filename for inspection).")
        collapsed = True

    epoch_ckpt_path = RL_CKPT_DIR / f"{RUN_NAME}_epoch_{epoch}_reward{val_reward:.4f}.pt"
    torch.save({"model": model.state_dict(), "epoch": epoch,
                "val_reward": val_reward, "val_per_task": val_per_task},
               epoch_ckpt_path)
    print(f"  saved epoch checkpoint -> {epoch_ckpt_path.name}")

    if val_reward > best_val_reward:
        best_val_reward = val_reward
        best_ckpt_path = epoch_ckpt_path
        print(f"  (new best, val_reward {val_reward:.4f})")

    if not collapsed:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "epoch": epoch, "best_val_reward": best_val_reward,
                    "best_ckpt_path": str(best_ckpt_path) if best_ckpt_path else None,
                    "history": history}, RESUME_PATH)
    else:
        break

if _collector is not None:
    _collector.close()
    print("ParallelRolloutCollector closed.")


## Cell 8 -- Training Curves

In [17]:
epochs_x = [h["epoch"] for h in history]
val_r    = [h["val_reward"] for h in history]
adv      = [h["mean_advantage"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_x, val_r, marker="o")
ax1.axhline(y=start_val_reward, color="gray", linestyle="--", label="starting policy")
ax1.set_xlabel("epoch"); ax1.set_ylabel("val reward (mean score)")
ax1.set_title("Validation reward over RL training"); ax1.legend()

ax2.plot(epochs_x, adv, marker="o", color="#55A868")
ax2.axhline(y=0, color="gray", linestyle="--")
ax2.set_xlabel("epoch"); ax2.set_ylabel("mean advantage (sample - greedy)")
ax2.set_title("Self-critical advantage over RL training")
plt.tight_layout()
plt.savefig(RL_CKPT_DIR / "rl_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 9 -- Sanity Checks vs. Starting Policy


In [18]:

_val_subset_path = RL_CKPT_DIR / "rl_pilot_val_subset.jsonl"
with open(_val_subset_path, "w", encoding="utf-8") as f:
    for d in val_records:
        f.write(json.dumps(d) + chr(10))

val_ds = SFTDataset(_val_subset_path, tokenizer, config)
_collate = lambda b: collate_sft(b, PAD_ID)
_val_loader_small = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=_collate)

policy_metrics = validate(model, _val_loader_small, device, PAD_ID)
ref_metrics    = validate(ref_model, _val_loader_small, device, PAD_ID)
print(f"Policy NLL after RL : {policy_metrics['nll']:.4f}  (p_gen {policy_metrics['p_gen']:.3f})")
print(f"Reference NLL (start): {ref_metrics['nll']:.4f}  (p_gen {ref_metrics['p_gen']:.3f})")
print(f"Delta: {policy_metrics['nll'] - ref_metrics['nll']:+.4f}")


In [19]:
N_SHOW = 2
_rng2 = random.Random(1)
for ttype in sorted(_val_by_type):
    sample = _rng2.sample(_val_by_type[ttype], min(N_SHOW, len(_val_by_type[ttype])))
    for d in sample:
        res = make_sft_sr(d, tokenizer, config)
        if res is None:
            continue
        enc_ids, _, task_id = res
        before_ids = tokenizer.encode(d["before"])
        after_ids  = tokenizer.encode(d["after"])
        nl_id = config.special_token_ids["newline"]
        gold_after = join_lines(split_lines_by_newline(after_ids, nl_id), nl_id)

        out = greedy_decode(model, enc_ids, task_id, device, max_new_tokens=MAX_NEW_TOKENS, collect_conf=False)
        pred = apply_sr(before_ids, out["tokens"], config)
        before_text = tokenizer.decode(before_ids)
        gold_text = tokenizer.decode(gold_after)
        pred_text = tokenizer.decode(pred) if pred is not None else "(apply_sr failed)"
        score = TASK_SCORERS[ttype](before_text, tokenizer.decode(pred) if pred is not None else None, gold_text)["score"]

        print("=" * 64)
        print(f"[{ttype.upper()}]  score={score:.3f}")
        print("=" * 64)
        print(f"BEFORE:\n{d['before']}")
        print(f"\nPRED AFTER:\n{pred_text}")
        print(f"\nGOLD AFTER:\n{d['after']}\n")


## Cell 10 -- Ref (pre-RL) vs. Policy (post-RL), same samples


In [20]:
N_SHOW2 = 2
_rng3 = random.Random(1)   # same seed as Cell 9's qualitative-samples cell -- same records

for ttype in sorted(_val_by_type):
    sample = _rng3.sample(_val_by_type[ttype], min(N_SHOW2, len(_val_by_type[ttype])))
    for d in sample:
        res = make_sft_sr(d, tokenizer, config)
        if res is None:
            continue
        enc_ids, _, task_id = res
        before_ids = tokenizer.encode(d["before"])
        after_ids  = tokenizer.encode(d["after"])
        nl_id = config.special_token_ids["newline"]
        gold_after = join_lines(split_lines_by_newline(after_ids, nl_id), nl_id)
        before_text = tokenizer.decode(before_ids)
        gold_text = tokenizer.decode(gold_after)

        out_ref = greedy_decode(ref_model, enc_ids, task_id, device,
                                 max_new_tokens=MAX_NEW_TOKENS, collect_conf=False)
        pred_ref = apply_sr(before_ids, out_ref["tokens"], config)
        pred_ref_text = tokenizer.decode(pred_ref) if pred_ref is not None else None
        score_ref = TASK_SCORERS[ttype](before_text, pred_ref_text, gold_text)["score"]

        out_policy = greedy_decode(model, enc_ids, task_id, device,
                                    max_new_tokens=MAX_NEW_TOKENS, collect_conf=False)
        pred_policy = apply_sr(before_ids, out_policy["tokens"], config)
        pred_policy_text = tokenizer.decode(pred_policy) if pred_policy is not None else None
        score_policy = TASK_SCORERS[ttype](before_text, pred_policy_text, gold_text)["score"]

        delta = score_policy - score_ref
        flag = "  <-- RL made this WORSE" if delta < -0.05 else ("  <-- RL made this better" if delta > 0.05 else "")

        print("=" * 72)
        print(f"[{ttype.upper()}]  ref_score={score_ref:.3f}  policy_score={score_policy:.3f}  "
              f"delta={delta:+.3f}{flag}")
        print("=" * 72)
        print(f"BEFORE:\n{d['before']}")
        print(f"\nREF (pre-RL) AFTER:\n{pred_ref_text if pred_ref_text is not None else '(apply_sr failed)'}")
        print(f"\nPOLICY (post-RL) AFTER:\n{pred_policy_text if pred_policy_text is not None else '(apply_sr failed)'}")
        print(f"\nGOLD AFTER:\n{d['after']}\n")
